# Model B: grounded GPT-2 (text + image-caption)
Same GPT-2 backbone as Model A, with a prepended grounding slot: a projected DINOv2 embedding for caption examples, a learned placeholder for plain text examples.

Upload `model_b_combined.json` to your Drive before running this (see path below). Runtime > Change runtime type > GPU, before running.

In [1]:
!pip install -q transformers datasets accelerate

In [3]:
from google.colab import drive
drive.mount('/content/drive')

# where model_b_combined.json lives on your drive, update if you put it elsewhere
COMBINED_DATA_PATH = '/content/drive/MyDrive/babylm_data/model_b_texts.json'
EMBEDDINGS_PATH = '/content/drive/MyDrive/babylm_data/model_b_embeddings.npy'

DRIVE_CHECKPOINT_DIR = '/content/drive/MyDrive/babylm_checkpoints/model_b'
LOCAL_CHECKPOINT_DIR = '/content/checkpoints/model_b'

# set True if you uploaded a checkpoint from the local mac run
# (fp32) and want to continue it here, set False for a fresh
# Colab-only run (uses fp16 for speed)
RESUMING_FROM_LOCAL_CHECKPOINT = False

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import os
import json
import shutil
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from transformers import GPT2Config, GPT2LMHeadModel, GPT2TokenizerFast
from transformers import Trainer, TrainingArguments, TrainerCallback
from transformers.modeling_outputs import CausalLMOutput

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"using device: {device}")

using device: cuda


In [5]:
SEQ_LEN = 256
EMBED_DIM = 768  # dinov2 output size

tokenizer = GPT2TokenizerFast.from_pretrained("BabyLM-community/BabyLM-2026-Baseline-GPT2-Strict-Small")
tokenizer.pad_token = tokenizer.eos_token

In [13]:
class GroundedGPT2(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.gpt2 = GPT2LMHeadModel(config)
        self.image_proj = nn.Linear(EMBED_DIM, config.n_embd)
        self.no_image_embed = nn.Parameter(torch.randn(config.n_embd) * 0.02)

    def forward(self, input_ids, attention_mask, labels=None, image_embedding=None, has_image=None):
        wte = self.gpt2.transformer.wte(input_ids)
        batch_size = input_ids.shape[0]
        no_image = self.no_image_embed.unsqueeze(0).expand(batch_size, -1)
        if image_embedding is not None and has_image is not None:
            projected = self.image_proj(image_embedding).to(no_image.dtype)
            mask = has_image.to(no_image.dtype).unsqueeze(-1)
            slot = mask * projected + (1 - mask) * no_image
        else:
            slot = no_image
        slot = slot.unsqueeze(1)

        inputs_embeds = torch.cat([slot, wte], dim=1)
        full_attention_mask = torch.cat(
            [torch.ones(batch_size, 1, device=attention_mask.device), attention_mask], dim=1
        )
        full_labels = None
        if labels is not None:
            ignore_col = torch.full((batch_size, 1), -100, device=labels.device, dtype=labels.dtype)
            full_labels = torch.cat([ignore_col, labels], dim=1)

        outputs = self.gpt2(inputs_embeds=inputs_embeds, attention_mask=full_attention_mask, labels=full_labels)
        return CausalLMOutput(loss=outputs.loss, logits=outputs.logits)

    def state_dict(self, *args, **kwargs):
      sd = super().state_dict(*args, **kwargs)
      # lm_head.weight is tied to wte.weight (same tensor), drop the
      # duplicate key so safetensors doesn't choke on shared memory
      sd.pop("gpt2.lm_head.weight", None)
      return sd

    def load_state_dict(self, state_dict, strict=True):
      # lm_head.weight will show as "missing" since we dropped it on
      # save, that's expected, tying restores it automatically
      return super().load_state_dict(state_dict, strict=False)

In [14]:
class GroundedDataset(Dataset):
    def __init__(self, texts, embeddings, tokenizer, seq_len, chunk_size=5000):
        text_list = [t["text"] for t in texts]
        n = len(text_list)

        self.input_ids = torch.zeros((n, seq_len), dtype=torch.long)
        self.attention_mask = torch.zeros((n, seq_len), dtype=torch.long)

        for start in range(0, n, chunk_size):
            end = min(start + chunk_size, n)
            enc = tokenizer(
                text_list[start:end], truncation=True, max_length=seq_len,
                padding="max_length", return_tensors="pt",
            )
            self.input_ids[start:end] = enc["input_ids"]
            self.attention_mask[start:end] = enc["attention_mask"]

        labels = self.input_ids.clone()
        labels[self.attention_mask == 0] = -100
        self.labels = labels

        self.has_image = torch.tensor([t["has_image"] for t in texts])
        self.embeddings = torch.from_numpy(embeddings).float()

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels": self.labels[idx],
            "image_embedding": self.embeddings[idx],
            "has_image": self.has_image[idx],
        }

In [15]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:            12Gi       7.6Gi       1.2Gi        14Mi       3.9Gi       4.7Gi
Swap:             0B          0B          0B


In [16]:
import numpy as np

print("loading combined dataset...")
texts = json.load(open(COMBINED_DATA_PATH))  # now points to model_b_texts.json
embeddings = np.load(EMBEDDINGS_PATH)         # new: path to model_b_embeddings.npy

total_words = sum(len(t["text"].split()) for t in texts)
print(f"total words: {total_words}")

split_idx = int(len(texts) * 0.98)
train_texts, eval_texts = texts[:split_idx], texts[split_idx:]
train_embeds, eval_embeds = embeddings[:split_idx], embeddings[split_idx:]

train_dataset = GroundedDataset(train_texts, train_embeds, tokenizer, SEQ_LEN)
eval_dataset = GroundedDataset(eval_texts, eval_embeds, tokenizer, SEQ_LEN)
print(f"train size: {len(train_dataset)}, eval size: {len(eval_dataset)}")

del texts, embeddings, train_texts, eval_texts, train_embeds, eval_embeds
import gc
gc.collect()

loading combined dataset...
total words: 3500000
train size: 293688, eval size: 5994


150

In [20]:
BATCH_SIZE = 16
GRAD_ACCUM = 2

# rough words/step estimate, same approach as model a, to convert
# the 1M-word checkpoint requirement into a step count
words_per_step = BATCH_SIZE * GRAD_ACCUM * (SEQ_LEN / 1.3)
steps_per_1M_words = max(1, round(1_000_000 / words_per_step))
print(f"approx steps per 1M words: {steps_per_1M_words}")

config = GPT2Config(
    vocab_size=tokenizer.vocab_size,
    n_positions=SEQ_LEN + 1,  # +1 for the grounding slot
    n_embd=768,
    n_layer=12,
    n_head=12,
    activation_function="gelu_new",
    bos_token_id=1,
    eos_token_id=2,
)
model = GroundedGPT2(config).to(device)
print(f"model params: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

approx steps per 1M words: 159
model params: 98.4M


In [21]:
# writes checkpoints to local disk (fast, reliable), then copies
# each finished one to drive and trims old ones there, same fix
# we used for model a's storage crashes
class SyncToDriveCallback(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
        ckpts = [d for d in os.listdir(LOCAL_CHECKPOINT_DIR) if d.startswith("checkpoint-")]
        if not ckpts:
            return
        latest = max(ckpts, key=lambda x: int(x.split("-")[1]))
        src = os.path.join(LOCAL_CHECKPOINT_DIR, latest)
        dst = os.path.join(DRIVE_CHECKPOINT_DIR, latest)

        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)

        drive_ckpts = sorted(
            [d for d in os.listdir(DRIVE_CHECKPOINT_DIR) if d.startswith("checkpoint-")],
            key=lambda x: int(x.split("-")[1])
        )
        for old in drive_ckpts[:-2]:
            shutil.rmtree(os.path.join(DRIVE_CHECKPOINT_DIR, old), ignore_errors=True)

In [19]:
args = TrainingArguments(
    output_dir=LOCAL_CHECKPOINT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=3,  # matches the local reduced-scope run
    save_steps=steps_per_1M_words,
    save_total_limit=2,
    logging_steps=50,
    report_to="none",
    # RESUMING_FROM_LOCAL_CHECKPOINT below controls this, don't
    # set fp16=True if resuming a checkpoint that was saved in fp32
    # on the mac, mismatched precision on resume can destabilize
    # the optimizer state
    fp16=not RESUMING_FROM_LOCAL_CHECKPOINT,

)

trainer = Trainer(model=model, args=args, train_dataset=train_dataset, eval_dataset=eval_dataset, callbacks=[SyncToDriveCallback()])

In [22]:
def find_latest_valid_checkpoint(checkpoint_dir):
    if not os.path.exists(checkpoint_dir):
        return None
    checkpoints = [d for d in os.listdir(checkpoint_dir) if d.startswith("checkpoint-")]
    checkpoints.sort(key=lambda x: int(x.split("-")[1]), reverse=True)
    required_files = ["trainer_state.json", "pytorch_model.bin", "optimizer.pt"]
    for ckpt in checkpoints:
        ckpt_path = os.path.join(checkpoint_dir, ckpt)
        if all(os.path.exists(os.path.join(ckpt_path, f)) for f in required_files):
            return ckpt_path
    return None

resume_path = find_latest_valid_checkpoint(DRIVE_CHECKPOINT_DIR)
print(f"resuming from: {resume_path}")

trainer.train(resume_from_checkpoint=resume_path)

resuming from: None


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
50,6.904523
100,5.610457
150,5.241500
200,4.993754
250,4.987678
300,4.872679
350,4.809475
400,4.668518
450,4.528144
500,4.593534


KeyboardInterrupt: 

In [ ]:
trainer.save_model(f"{DRIVE_CHECKPOINT_DIR}/final")
tokenizer.save_pretrained(f"{DRIVE_CHECKPOINT_DIR}/final")
print("done, final model saved to drive")